In [17]:
# Bloco 1 - Importar a base de monitores com cidade e estado

from pathlib import Path
import pandas as pd


def obter_pasta_do_notebook():
    return Path.cwd()


def importar_csv_mesma_pasta(nome_arquivo, nome_base, colunas=None, sep=",", engine=None):
    try:
        pasta_notebook = obter_pasta_do_notebook()
        caminho_arquivo = pasta_notebook / nome_arquivo

        argumentos = {
            "filepath_or_buffer": caminho_arquivo,
            "sep": sep,
            "encoding": "utf-8-sig",
            "usecols": colunas
        }

        if engine is not None:
            argumentos["engine"] = engine
        else:
            argumentos["low_memory"] = False

        df = pd.read_csv(**argumentos)
        df.columns = df.columns.str.strip()

        print(f"Base '{nome_base}' importada com sucesso!")
        print(f"Arquivo importado: {caminho_arquivo}")
        print(f"Quantidade de linhas: {df.shape[0]}")
        print(f"Quantidade de colunas: {df.shape[1]}")
        print()

        return df

    except FileNotFoundError:
        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")
        return None

    except Exception as erro:
        print(f"Erro ao importar a base '{nome_base}'.")
        print(f"Detalhes do erro: {erro}")
        return None


def inserir_cidade_estado(df_monitores, df_municipios):

    df_monitores = df_monitores.copy()
    df_municipios = df_municipios.copy()

    df_monitores.columns = df_monitores.columns.str.strip()
    df_municipios.columns = df_municipios.columns.str.strip()

    colunas_municipios_necessarias = [
        "Código Município Completo",
        "Nome_Município",
        "Nome_UF"
    ]

    colunas_faltantes = [
        coluna
        for coluna in colunas_municipios_necessarias
        if coluna not in df_municipios.columns
    ]

    if colunas_faltantes:
        print("Erro nas colunas da base de municípios.")
        print(colunas_faltantes)
        return None

    df_monitores["codigo_cidade"] = pd.to_numeric(
        df_monitores["codigo_cidade"],
        errors="coerce"
    ).astype("Int64")

    df_municipios["Código Município Completo"] = pd.to_numeric(
        df_municipios["Código Município Completo"],
        errors="coerce"
    ).astype("Int64")

    municipios = (
        df_municipios[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ]
        .drop_duplicates(subset=["Código Município Completo"])
        .rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )
    )

    df_monitores = df_monitores.merge(
        municipios,
        on="codigo_cidade",
        how="left"
    )

    colunas = list(df_monitores.columns)

    colunas.remove("nome_cidade")
    colunas.remove("estado")

    posicao = colunas.index("codigo_cidade")

    colunas.insert(posicao + 1, "nome_cidade")
    colunas.insert(posicao + 2, "estado")

    df_monitores = df_monitores[colunas]

    print("Cidade e estado inseridos com sucesso!")

    print(
        "Registros sem município:",
        df_monitores["nome_cidade"].isna().sum()
    )

    return df_monitores


nome_arquivo_monitores = "script_05_monitores.csv"
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

colunas_monitores = [
    "codigo_cidade",
    "idade",
    "sexo",
    "vinculo",
    "salario",
    "turno_manha",
    "turno_tarde",
    "turno_noite",
    "loc_latitude",
    "loc_longitude",
    "dt_criacao",
    "dt_alteracao"
]

df_monitores = importar_csv_mesma_pasta(
    nome_arquivo_monitores,
    "monitores",
    colunas=colunas_monitores,
    sep="," 
)

df_municipios = importar_csv_mesma_pasta(
    nome_arquivo_municipios,
    "municípios",
    sep=None,
    engine="python"
)

if df_monitores is not None and df_municipios is not None:

    df_monitores_transformado = inserir_cidade_estado(
        df_monitores,
        df_municipios
    )

    if df_monitores_transformado is not None:
        display(df_monitores_transformado)

        # Exporta o DataFrame transformado para CSV na mesma pasta do notebook
        pasta_notebook = obter_pasta_do_notebook()
        nome_arquivo_saida = "script_05_monitores_consolidado.csv"
        caminho_arquivo_saida = pasta_notebook / nome_arquivo_saida

        df_monitores_transformado.to_csv(
            caminho_arquivo_saida,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\nArquivo exportado com sucesso!")
        print(f"Caminho: {caminho_arquivo_saida}")


Base 'monitores' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_05_monitores.csv
Quantidade de linhas: 22678
Quantidade de colunas: 12

Base 'municípios' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Cidade e estado inseridos com sucesso!
Registros sem município: 0


,codigo_cidade,nome_cidade,estado,idade,sexo,vinculo,salario,turno_manha,turno_tarde,turno_noite,loc_latitude,loc_longitude,dt_criacao,dt_alteracao
0,2300101,Abaiara,Ceará,22,1,4,750.00,S,N,N,NaN,NaN,2025-10-13 07:35:49,NaN
1,1705508,Colinas do Tocantins,Tocantins,43,2,3,NaN,S,N,N,NaN,NaN,2025-02-26 07:41:40,NaN
2,4321436,Terra de Areia,Rio Grande do Sul,34,1,2,NaN,S,S,N,NaN,NaN,2025-10-14 08:00:48,NaN
3,3159605,Santa Rita do Sapucaí,Minas Gerais,51,2,3,NaN,S,S,N,NaN,NaN,2025-03-28 09:56:40,NaN
4,2102200,Buriti,Maranhão,39,2,4,NaN,N,S,N,NaN,NaN,2025-04-04 15:13:31,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22673,2612208,Salgueiro,Pernambuco,41,2,3,NaN,S,S,N,NaN,NaN,2026-04-13 13:17:50,NaN
22674,4125506,São José dos Pinhais,Paraná,48,2,3,1571.56,S,S,N,NaN,NaN,2026-04-13 13:45:50,NaN
22675,4125506,São José dos Pinhais,Paraná,34,2,3,1571.56,S,S,N,NaN,NaN,2026-04-13 14:02:04,NaN
22676,4109005,Guapirama,Paraná,29,2,1,NaN,S,N,N,NaN,NaN,2026-04-13 14:02:54,NaN



Arquivo exportado com sucesso!
Caminho: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_05_monitores_consolidado.csv


In [18]:
# Bloco 2 - Preparar a base de monitores para consolidação

df_consolidacao = df_monitores_transformado.copy()

df_consolidacao["estado"] = (
    df_consolidacao["estado"]
    .fillna("Não identificado")
)

# Converter campos numéricos
colunas_numericas = [
    "sexo",
    "vinculo"
]

for coluna in colunas_numericas:
    df_consolidacao[coluna] = pd.to_numeric(
        df_consolidacao[coluna],
        errors="coerce"
    ).astype("Int64")

# Padronizar turnos
colunas_turno = [
    "turno_manha",
    "turno_tarde",
    "turno_noite"
]

for coluna in colunas_turno:
    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

print("Base de monitores preparada para consolidação.")

display(df_consolidacao.head())

Base de monitores preparada para consolidação.


,codigo_cidade,nome_cidade,estado,idade,sexo,vinculo,salario,turno_manha,turno_tarde,turno_noite,loc_latitude,loc_longitude,dt_criacao,dt_alteracao
0,2300101,Abaiara,Ceará,22,1,4,750.0,S,N,N,NaN,NaN,2025-10-13 07:35:49,NaN
1,1705508,Colinas do Tocantins,Tocantins,43,2,3,NaN,S,N,N,NaN,NaN,2025-02-26 07:41:40,NaN
2,4321436,Terra de Areia,Rio Grande do Sul,34,1,2,NaN,S,S,N,NaN,NaN,2025-10-14 08:00:48,NaN
3,3159605,Santa Rita do Sapucaí,Minas Gerais,51,2,3,NaN,S,S,N,NaN,NaN,2025-03-28 09:56:40,NaN
4,2102200,Buriti,Maranhão,39,2,4,NaN,N,S,N,NaN,NaN,2025-04-04 15:13:31,NaN


In [19]:
# Bloco 3 - Função para consolidar vínculo por estado

def consolidar_vinculo_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_monitores=("codigo_cidade", "size"),
            servidor_efetivo=("vinculo", lambda x: (x == 1).sum()),
            servidor_comissionado=("vinculo", lambda x: (x == 2).sum()),
            terceirizado=("vinculo", lambda x: (x == 3).sum()),
            outro_vinculo=("vinculo", lambda x: (x == 4).sum()),
            vinculo_nao_informado=("vinculo", lambda x: (~x.isin([1,2,3,4])).sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de vínculo por estado:")

    display(tabela)

    return tabela

In [20]:
# Bloco 4 - Consolidar vínculo

tabela_vinculo = consolidar_vinculo_por_estado(
    df_consolidacao
)

Tabela consolidada de vínculo por estado:


,estado,total_monitores,servidor_efetivo,servidor_comissionado,terceirizado,outro_vinculo,vinculo_nao_informado
0,Acre,99,4,0,69,26,0
1,Alagoas,1150,19,60,167,904,0
2,Amapá,51,0,9,0,42,0
3,Amazonas,95,0,29,29,37,0
4,Bahia,1783,66,125,768,824,0
5,Ceará,1624,61,78,61,1424,0
6,Espírito Santo,461,23,26,346,66,0
7,Goiás,283,58,60,96,69,0
8,Maranhão,1223,53,123,272,775,0
9,Mato Grosso,303,57,18,148,80,0


In [21]:
# Bloco 5 - Função para consolidar sexo por estado
# sexo: 1 - Masculino; 2 - Feminino; 3 - Não informado

def consolidar_sexo_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_monitores=("codigo_cidade", "size"),
            masculino=("sexo", lambda x: (x == 1).sum()),
            feminino=("sexo", lambda x: (x == 2).sum()),
            nao_informado=("sexo", lambda x: (x == 3).sum()),
            sexo_nao_identificado=("sexo", lambda x: (~x.isin([1, 2, 3])).sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de sexo por estado:")

    display(tabela)

    return tabela


tabela_sexo = consolidar_sexo_por_estado(
    df_consolidacao
)

Tabela consolidada de sexo por estado:


,estado,total_monitores,masculino,feminino,nao_informado,sexo_nao_identificado
0,Acre,99,37,62,0,0
1,Alagoas,1150,279,867,4,0
2,Amapá,51,15,36,0,0
3,Amazonas,95,49,46,0,0
4,Bahia,1783,295,1487,1,0
5,Ceará,1624,411,1209,4,0
6,Espírito Santo,461,31,430,0,0
7,Goiás,283,64,219,0,0
8,Maranhão,1223,345,872,6,0
9,Mato Grosso,303,26,277,0,0


In [22]:
# Bloco 7 - Função para consolidar turnos por estado

def consolidar_turnos_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_monitores=("codigo_cidade", "size"),
            turno_manha=("turno_manha", lambda x: (x == "S").sum()),
            turno_tarde=("turno_tarde", lambda x: (x == "S").sum()),
            turno_noite=("turno_noite", lambda x: (x == "S").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de turnos por estado:")

    display(tabela)

    return tabela

In [23]:
# Bloco 8 - Consolidar turnos

tabela_turnos = consolidar_turnos_por_estado(
    df_consolidacao
)

Tabela consolidada de turnos por estado:


,estado,total_monitores,turno_manha,turno_tarde,turno_noite
0,Acre,99,96,71,2
1,Alagoas,1150,1040,1014,47
2,Amapá,51,49,50,0
3,Amazonas,95,79,66,6
4,Bahia,1783,1501,1225,11
5,Ceará,1624,1339,1161,1
6,Espírito Santo,461,439,384,2
7,Goiás,283,221,212,2
8,Maranhão,1223,978,924,34
9,Mato Grosso,303,220,235,2


In [24]:
# Bloco 11 - Consolidar todas as tabelas e exportar para Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):

    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):

        tabela_temp = tabela.copy()

        if indice > 0 and "total_monitores" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_monitores"])

        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    tabela_final = (
        tabela_final
        .sort_values("estado")
        .reset_index(drop=True)
    )

    linha_brasil = {"estado": "Brasil"}

    for coluna in tabela_final.columns:

        if coluna != "estado":
            linha_brasil[coluna] = tabela_final[coluna].sum()

    tabela_final = pd.concat(
        [
            tabela_final,
            pd.DataFrame([linha_brasil])
        ],
        ignore_index=True
    )

    return tabela_final


tabelas_para_consolidar = [
    tabela_vinculo,
    tabela_sexo,
    tabela_turnos
]

tabela_consolidada_final = consolidar_tabelas_por_estado(
    tabelas_para_consolidar
)

pasta_notebook = obter_pasta_do_notebook()

nome_arquivo_excel = "Consolidação 05 - script_05_monitores.xlsx"

caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")

print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)

Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 05 - script_05_monitores.xlsx


,estado,total_monitores,servidor_efetivo,servidor_comissionado,terceirizado,outro_vinculo,vinculo_nao_informado,masculino,feminino,nao_informado,sexo_nao_identificado,turno_manha,turno_tarde,turno_noite
0,Acre,99,4,0,69,26,0,37,62,0,0,96,71,2
1,Alagoas,1150,19,60,167,904,0,279,867,4,0,1040,1014,47
2,Amapá,51,0,9,0,42,0,15,36,0,0,49,50,0
3,Amazonas,95,0,29,29,37,0,49,46,0,0,79,66,6
4,Bahia,1783,66,125,768,824,0,295,1487,1,0,1501,1225,11
5,Ceará,1624,61,78,61,1424,0,411,1209,4,0,1339,1161,1
6,Espírito Santo,461,23,26,346,66,0,31,430,0,0,439,384,2
7,Goiás,283,58,60,96,69,0,64,219,0,0,221,212,2
8,Maranhão,1223,53,123,272,775,0,345,872,6,0,978,924,34
9,Mato Grosso,303,57,18,148,80,0,26,277,0,0,220,235,2
